# Figuras del Capítulo 5 — Interpretabilidad Mecanística

Genera todas las figuras (versiones `es` y `en`) del capítulo 5 a partir de los
CSVs exportados por los notebooks 04 (`results/notebook4/`) y 06
(`results/notebook6/`).

**Figuras producidas (en `figures/`):**
1. `cap5_probing_heatmap_{es,en}.png` — F1 probes 23 × 13, marker de cristalización
2. `cap5_information_gain_{es,en}.png` — ganancia informativa incremental por capa
3. `cap5_crystal_correlations_{es,en}.png` — cristalización vs F1 máx. | freq. vs cristalización
4. `cap5_head_categories_{es,en}.png` — distribución de categorías de cabezas por capa

Los títulos se delegan al `\caption{}` de LaTeX (convención TFG).

## Setup

In [ ]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/transformer-structural-compression-v2'
    os.chdir(PROJECT_ROOT)
    sys.path.insert(0, PROJECT_ROOT)
    PLOTS_DIR = os.path.join(PROJECT_ROOT, 'plots')
else:
    # Local: notebook lives in plots/, project root is one level up
    cwd = os.getcwd()
    if os.path.basename(cwd) == 'plots':
        PLOTS_DIR = cwd
        PROJECT_ROOT = os.path.abspath(os.path.join(cwd, '..'))
    else:
        PROJECT_ROOT = cwd
        PLOTS_DIR = os.path.join(PROJECT_ROOT, 'plots')

if PLOTS_DIR not in sys.path:
    sys.path.insert(0, PLOTS_DIR)

os.chdir(PLOTS_DIR)

print(f'Project root: {PROJECT_ROOT}')
print(f'Plots dir:    {PLOTS_DIR}')
print(f'CWD:          {os.getcwd()}')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch
from scipy.stats import spearmanr

import tfg_plot_style as sty
sty.apply(lang='es')

# CSV paths (refactored NB04/NB06 outputs)
NB04_DIR = os.path.join(PROJECT_ROOT, 'results', 'notebook4')
NB06_DIR = os.path.join(PROJECT_ROOT, 'results', 'notebook6')

# NB04 outputs
CSV_PROBE_WIDE    = os.path.join(NB04_DIR, 'probe_results.csv')
CSV_CRYSTAL       = os.path.join(NB04_DIR, 'crystallization_layers.csv')
CSV_INFO_GAIN     = os.path.join(NB04_DIR, 'layer_information_gain.csv')
CSV_EMO_FREQ      = os.path.join(NB04_DIR, 'emotion_frequency.csv')
CSV_PROBE_CORR    = os.path.join(NB04_DIR, 'probing_correlations.csv')

# NB06 outputs
CSV_HEAD_CATS     = os.path.join(NB06_DIR, 'head_categories.csv')

os.makedirs('figures', exist_ok=True)


def load_csv(path, fallback=None, name=''):
    try:
        df = pd.read_csv(path)
        print(f'[ok] {name}: {path} -- {df.shape}')
        return df
    except FileNotFoundError:
        if fallback is not None:
            print(f'[!]  {name}: not found -- using fallback')
            return fallback
        print(f'[X]  {name}: {path} -- NOT FOUND')
        return None

## Figura 16 — Heatmap de probes con marcadores de cristalización

NB04 `probe_results.csv`: matriz (emoción × capa) con F1. Primera columna
`emotion`. `crystallization_layers.csv` contiene la capa de cristalización por
emoción. Emociones ordenadas por profundidad de cristalización creciente.

In [ ]:
probe_df   = load_csv(CSV_PROBE_WIDE, name='probe F1 (wide)')
crystal_df = load_csv(CSV_CRYSTAL, name='crystallization')

if probe_df is not None and crystal_df is not None:
    # Ensure emotion column is the index
    probe_df = probe_df.set_index(probe_df.columns[0])
    layer_labels = list(probe_df.columns)
    n_layers = len(layer_labels)

    # Build integer layer index 0..N-1 for marker placement
    # NB04 uses labels like 'embedding', 'layer_0', ..., 'layer_11'
    layer_idx_of = {name: i for i, name in enumerate(layer_labels)}

    crystal_df = crystal_df.copy()
    crystal_df = crystal_df.set_index('emotion')

    # Order emotions ascending by crystal layer (then by max F1 desc as tiebreak)
    ordered = crystal_df.sort_values(
        ['crystallization_layer', 'max_probe_f1'],
        ascending=[True, False],
    ).index.tolist()

    # Intersect with probe_df rows to keep only emotions present in both
    ordered = [e for e in ordered if e in probe_df.index]
    probe_mat = probe_df.loc[ordered]

    def plot_probing_heatmap(ax, df=probe_mat, cr=crystal_df, order=ordered, n=n_layers):
        sns.heatmap(df, cmap='YlOrRd', vmin=0, vmax=1,
                    linewidths=0.5, linecolor=sty.BG,
                    cbar_kws={'label': sty.L['probe_f1'], 'shrink': 0.8},
                    ax=ax)

        cbar = ax.collections[0].colorbar
        cbar.ax.tick_params(labelsize=sty.TICK_SIZE)
        cbar.set_label(sty.L['probe_f1'], fontsize=sty.LABEL_SIZE)

        # Crystallization markers (white circle with dark outline)
        for row_i, emo in enumerate(order):
            cl = cr.loc[emo, 'crystallization_layer']
            try:
                col_i = int(cl)
            except (TypeError, ValueError):
                col_i = None
            if col_i is None or col_i < 0 or col_i >= n:
                continue
            ax.scatter(col_i + 0.5, row_i + 0.5, s=38,
                       facecolor='white', edgecolor=sty.INK,
                       linewidths=1.2, zorder=5)

        # Cosmetic axis labels (numeric layer ticks if names are layer_*/embedding)
        xlabels = []
        for name in df.columns:
            if name == 'embedding':
                xlabels.append('E')
            elif name.startswith('layer_'):
                xlabels.append(name.split('_')[1])
            else:
                xlabels.append(str(name))
        ax.set_xticklabels(xlabels, rotation=0, fontsize=sty.TICK_SIZE)
        ax.set_yticklabels([sty.EMOTION_DISPLAY.get(e, e.capitalize()) for e in order],
                           rotation=0, fontsize=7.5)

        ax.set_xlabel(sty.L['layer'])
        ax.set_ylabel(sty.L['emotion'])

    sty.generate_both(plot_probing_heatmap, 'probing_heatmap',
                      chapter=5, figsize=(6.8, 8.2))
else:
    print('-> Run NB04 export to produce probe_results.csv and crystallization_layers.csv')

## Figura 17 — Ganancia informativa incremental por capa

NB04 `layer_information_gain.csv` columnas: `layer_idx, layer_name,
mean_probe_f1, information_gain`. La ganancia de la primera capa queda como
NaN (no hay capa anterior); el plot la trata como 0.

In [ ]:
info_df = load_csv(CSV_INFO_GAIN, name='layer information gain')

if info_df is not None:
    def plot_info_gain(ax, df=info_df):
        df = df.copy()
        df['information_gain'] = df['information_gain'].fillna(0.0)

        x = df['layer_idx'].values
        gains = df['information_gain'].values
        means = df['mean_probe_f1'].values

        # Bar chart for incremental gain (primary axis)
        colors = [sty.SAGE if g >= 0 else sty.TERRA for g in gains]
        bars = ax.bar(x, gains, color=colors, edgecolor=sty.BG,
                      linewidth=0.7, width=0.7, zorder=3,
                      label=sty.L['info_gain'])

        # Zero line
        ax.axhline(y=0, color=sty.INK_3, lw=0.7, alpha=0.6, zorder=2)

        ax.set_xlabel(sty.L['layer'])
        ax.set_ylabel(sty.L['info_gain'])

        # Layer tick labels
        xt_labels = []
        for name in df['layer_name']:
            if name == 'embedding':
                xt_labels.append('E')
            elif name.startswith('layer_'):
                xt_labels.append(name.split('_')[1])
            else:
                xt_labels.append(str(name))
        ax.set_xticks(x)
        ax.set_xticklabels(xt_labels)

        # Overlay mean F1 line (secondary axis)
        ax2 = ax.twinx()
        ax2.plot(x, means, color=sty.BLUE, marker='o', markersize=5,
                 lw=1.8, markeredgecolor='white', markeredgewidth=0.8,
                 label=sty.L['mean_probe_f1'], zorder=5)
        ax2.set_ylabel(sty.L['mean_probe_f1'], color=sty.BLUE)
        ax2.tick_params(axis='y', labelcolor=sty.BLUE, labelsize=sty.TICK_SIZE)
        ax2.set_ylim(0, max(1.0, float(np.nanmax(means)) * 1.05))
        ax2.grid(False)
        ax2.spines['top'].set_visible(False)

        # Annotate the biggest jump
        valid_idx = np.argmax(gains)
        if gains[valid_idx] > 0:
            ax.annotate(f'+{gains[valid_idx]:.3f}',
                        (x[valid_idx], gains[valid_idx]),
                        textcoords='offset points', xytext=(0, 6),
                        ha='center', fontsize=sty.ANNOTATION_SIZE,
                        color=sty.INK_2)

        # Combined legend
        h1, l1 = ax.get_legend_handles_labels()
        h2, l2 = ax2.get_legend_handles_labels()
        ax.legend(h1 + h2, l1 + l2, loc='upper left', fontsize=sty.LEGEND_SIZE)

    sty.generate_both(plot_info_gain, 'information_gain',
                      chapter=5, figsize=sty.FIG_WIDE)
else:
    print('-> Run NB04 export to produce layer_information_gain.csv')

## Figura 18 — Correlaciones de cristalización

Izquierda: capa de cristalización vs. F1 máximo.
Derecha: frecuencia en el dataset vs. capa de cristalización.

Usa `crystallization_layers.csv` (incluye `train_frequency` y `max_probe_f1`).
Si existe `probing_correlations.csv` se usan sus ρ/p exactos; si no, se
calculan con `scipy.stats.spearmanr`.

In [ ]:
crystal_df = load_csv(CSV_CRYSTAL, name='crystallization (corr panel)')
corr_df    = load_csv(CSV_PROBE_CORR, name='probing correlations')

if crystal_df is not None:
    crystal_df = crystal_df.copy()

    # Compute spearman correlations locally as fallback / verification
    cl = crystal_df['crystallization_layer'].values.astype(float)
    mf = crystal_df['max_probe_f1'].values.astype(float)
    fq = crystal_df['train_frequency'].values.astype(float)

    def _rho_p(df, relation, fallback_x, fallback_y):
        if df is not None and 'relation' in df.columns:
            row = df[df['relation'] == relation]
            if len(row):
                return float(row.iloc[0]['spearman_r']), float(row.iloc[0]['p_value'])
        r, p = spearmanr(fallback_x, fallback_y)
        return float(r), float(p)

    rho_depth, p_depth = _rho_p(corr_df, 'crystal_layer_vs_max_probe_f1', cl, mf)
    rho_freq,  p_freq  = _rho_p(corr_df, 'frequency_vs_crystal_layer', fq, cl)

    def _format_p(p):
        if p < 0.001:
            return 'p < 0{,}001' if sty.L is sty.LABELS_ES else 'p < 0.001'
        return f'p = {p:.3f}'.replace('.', '{,}') if sty.L is sty.LABELS_ES else f'p = {p:.3f}'

    def plot_correlations(df=crystal_df, rho_d=rho_depth, p_d=p_depth,
                          rho_f=rho_freq, p_f=p_freq):
        fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.5))

        # --- Left: crystal layer vs max F1 ---
        ax = axes[0]
        ax.scatter(df['crystallization_layer'], df['max_probe_f1'],
                   s=55, color=sty.BLUE, edgecolor='white', linewidth=0.8,
                   alpha=0.85, zorder=5)

        # Regression trend line
        x_reg = df['crystallization_layer'].values.astype(float)
        y_reg = df['max_probe_f1'].values.astype(float)
        if len(x_reg) >= 2 and np.std(x_reg) > 0:
            coef = np.polyfit(x_reg, y_reg, 1)
            xs = np.linspace(x_reg.min(), x_reg.max(), 50)
            ax.plot(xs, np.polyval(coef, xs), color=sty.INK_3, lw=1.2,
                    ls='--', alpha=0.6, zorder=3)

        ax.set_xlabel(sty.L['crystal_layer'])
        ax.set_ylabel(sty.L['max_probe_f1'])
        sign = '-' if rho_d < 0 else '+'
        txt = f'$\\rho$ = {sign}{abs(rho_d):.3f}\n{_format_p(p_d)}'
        ax.text(0.05, 0.05, txt, transform=ax.transAxes,
                fontsize=sty.ANNOTATION_SIZE, color=sty.INK_2,
                va='bottom', ha='left',
                bbox=dict(boxstyle='round,pad=0.35', facecolor=sty.BG,
                          edgecolor=sty.SPINE, alpha=0.9, lw=0.5))

        # --- Right: freq vs crystal layer ---
        ax = axes[1]
        # Use log scale on x if frequency spans orders of magnitude
        use_log = (fq.max() / max(fq.min(), 1.0)) > 50

        ax.scatter(df['train_frequency'], df['crystallization_layer'],
                   s=55, color=sty.TERRA, edgecolor='white', linewidth=0.8,
                   alpha=0.85, zorder=5)

        if use_log:
            ax.set_xscale('log')

        ax.set_xlabel(sty.L['dataset_freq'])
        ax.set_ylabel(sty.L['crystal_layer'])
        sign = '-' if rho_f < 0 else '+'
        txt = f'$\\rho$ = {sign}{abs(rho_f):.3f}\n{_format_p(p_f)}'
        ax.text(0.05, 0.95, txt, transform=ax.transAxes,
                fontsize=sty.ANNOTATION_SIZE, color=sty.INK_2,
                va='top', ha='left',
                bbox=dict(boxstyle='round,pad=0.35', facecolor=sty.BG,
                          edgecolor=sty.SPINE, alpha=0.9, lw=0.5))

        plt.tight_layout()
        return fig

    # Dual-language manual render (two-panel doesn't fit generate_both's 1 ax signature)
    for lang in ['es', 'en']:
        sty.apply(lang=lang)
        fig = plot_correlations()
        sty.save(fig, 'crystal_correlations', chapter=5, lang=lang)
else:
    print('-> Run NB04 export to produce crystallization_layers.csv')

## Figura 19 — Distribución de categorías de cabezas por capa

NB06 `head_categories.csv` columnas: `layer, head, category, ...`. Categorías:
`Critical Specialists`, `Critical Generalists`, `Minor Specialists`,
`Dispensable`. Barras apiladas mostrando cuántas cabezas (de 12) caen en cada
categoría por capa.

In [ ]:
cat_df = load_csv(CSV_HEAD_CATS, name='head categories')

if cat_df is not None:
    CAT_ORDER = ['Critical Specialists', 'Critical Generalists',
                 'Minor Specialists', 'Dispensable']

    # Count (layer x category)
    counts = (cat_df.groupby(['layer', 'category']).size()
              .unstack(fill_value=0))
    # Reindex to canonical order, filling missing categories with 0
    for c in CAT_ORDER:
        if c not in counts.columns:
            counts[c] = 0
    counts = counts[CAT_ORDER]
    counts = counts.reindex(sorted(counts.index)).fillna(0).astype(int)

    def plot_head_categories(ax, df=counts, order=CAT_ORDER):
        layers = df.index.values
        x = np.arange(len(layers))

        cat_colors = {
            'Critical Specialists':  sty.TERRA,
            'Critical Generalists':  sty.SAND,
            'Minor Specialists':     sty.BLUE_L,
            'Dispensable':           sty.INK_3,
        }
        cat_labels_key = {
            'Critical Specialists': 'critical_specialists',
            'Critical Generalists': 'critical_generalists',
            'Minor Specialists':    'minor_specialists',
            'Dispensable':          'dispensable',
        }

        bottom = np.zeros(len(layers))
        for cat in order:
            vals = df[cat].values.astype(float)
            ax.bar(x, vals, bottom=bottom,
                   color=cat_colors[cat],
                   edgecolor=sty.BG, linewidth=0.7,
                   label=sty.L[cat_labels_key[cat]])
            # Annotate non-zero counts
            for xi, v, b in zip(x, vals, bottom):
                if v >= 1:
                    ax.text(xi, b + v / 2, f'{int(v)}',
                            ha='center', va='center',
                            fontsize=sty.SMALL_SIZE,
                            color='white' if cat in ('Critical Specialists', 'Dispensable') else sty.INK)
            bottom += vals

        ax.set_xticks(x)
        ax.set_xticklabels([str(l) for l in layers])
        ax.set_xlabel(sty.L['layer'])
        ax.set_ylabel(sty.L['head_count'])
        ax.set_ylim(0, 13)
        ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.18),
                  ncol=4, fontsize=sty.LEGEND_SIZE, frameon=False)
        ax.grid(axis='x', visible=False)

    sty.generate_both(plot_head_categories, 'head_categories',
                      chapter=5, figsize=(7.2, 4.2))
else:
    print('-> Run NB06 export to produce head_categories.csv')

## Figura 20 / 21 — Logit lens (datos compartidos)

Las dos figuras siguientes (Figura 20: probe vs logit lens, y Figura 21: la
U-curve agregada) se construyen sobre las **mismas activaciones**. Esta celda
las computa desde cero: carga el dataset GoEmotions, filtra a 100 frases por
emoción mono-etiquetadas (≈ 2.300 en total), ejecuta el modelo
`23emo-final` para obtener el [CLS] de cada capa, y aplica el pooler +
classifier reales (logit lens) sobre cada capa.

**No depende del cache de `viz/`** — la extracción es íntegra y
reproducible desde el modelo y el dataset HuggingFace.

In [ ]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_CHECKPOINT = os.path.join(PROJECT_ROOT, 'results', 'checkpoints', '23emo-final')
N_PER_EMOTION    = 100      # ≈ 2.300 frases en total
MAX_LEN          = 64
BATCH_SIZE       = 32

EMOTIONS_23 = [
    'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring',
    'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval',
    'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'joy',
    'love', 'optimism', 'realization', 'remorse', 'sadness', 'surprise',
]
EXCLUDED = {'neutral', 'grief', 'nervousness', 'pride', 'relief'}

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

# ─── Load model + tokenizer ───────────────────────────────────────────────
print('Loading tokenizer (bert-base-uncased)...')
tok = AutoTokenizer.from_pretrained('bert-base-uncased')
print(f'Loading model ({MODEL_CHECKPOINT})...')
mdl = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT, output_hidden_states=True
).to(device).eval()

# ─── Sample test sentences (single-label, 100 per emotion) ────────────────
print('Loading GoEmotions raw split...')
ds = load_dataset('go_emotions', 'raw', split='train')
rng = np.random.default_rng(42)
indices = rng.permutation(len(ds))

selected = []
counts = {e: 0 for e in EMOTIONS_23}
for idx in indices:
    if all(c >= N_PER_EMOTION for c in counts.values()):
        break
    ex = ds[int(idx)]
    if any(ex.get(e, 0) == 1 for e in EXCLUDED):
        continue
    active = [e for e in EMOTIONS_23 if ex.get(e, 0) == 1]
    if len(active) != 1:
        continue
    emo = active[0]
    if counts[emo] >= N_PER_EMOTION:
        continue
    counts[emo] += 1
    selected.append({'text': ex['text'], 'label': emo})
print(f'Selected {len(selected)} single-label sentences')

# ─── Forward pass — extract CLS per layer ─────────────────────────────────
all_cls = []
labels  = []
with torch.no_grad():
    n_batches = (len(selected) + BATCH_SIZE - 1) // BATCH_SIZE
    for i in range(0, len(selected), BATCH_SIZE):
        batch = selected[i:i + BATCH_SIZE]
        enc = tok([b['text'] for b in batch], padding=True, truncation=True,
                  max_length=MAX_LEN, return_tensors='pt').to(device)
        out = mdl(**enc)
        hidden = torch.stack(out.hidden_states, dim=1)        # (B, 13, T, 768)
        cls = hidden[:, :, 0, :].cpu().numpy()                # (B, 13, 768)
        all_cls.append(cls)
        labels.extend(b['label'] for b in batch)
        if (i // BATCH_SIZE) % 10 == 0:
            print(f'  batch {i // BATCH_SIZE + 1}/{n_batches}')

cls_arr   = np.concatenate(all_cls, axis=0).astype(np.float32)  # (N, 13, 768)
label_arr = np.array(labels)
N         = len(label_arr)
print(f'CLS array: {cls_arr.shape}')

# ─── Logit lens (pooler + classifier reales aplicados a cada capa) ────────
Wp = mdl.bert.pooler.dense.weight.detach().cpu().numpy()
bp = mdl.bert.pooler.dense.bias.detach().cpu().numpy()
Wc = mdl.classifier.weight.detach().cpu().numpy()
bc = mdl.classifier.bias.detach().cpu().numpy()

pooled   = np.tanh(np.einsum('slh,ph->slp', cls_arr, Wp) + bp)   # (N, 13, 768)
logits   = np.einsum('slp,ep->sle', pooled, Wc) + bc             # (N, 13, 23)
sigmoids = 1.0 / (1.0 + np.exp(-logits))                         # (N, 13, 23)

label_to_idx = {e: i for i, e in enumerate(EMOTIONS_23)}
y = np.array([label_to_idx[l] for l in label_arr])

# Per-sentence gold sigmoid (used by Figura 20 to group by crystallization)
gold_sig = sigmoids[np.arange(N), :, y]                          # (N, 13)

# Aggregate curves (used by Figura 21 — the U-curve)
top1_per_layer = sigmoids.max(axis=2).mean(axis=0)
gold_per_layer = sigmoids[np.arange(N), :, y].mean(axis=0)
sum_per_layer  = sigmoids.sum(axis=2).mean(axis=0)

# Free GPU memory
del mdl, hidden
if device == 'cuda':
    torch.cuda.empty_cache()

print(f'top-1 range: {top1_per_layer.min():.3f} – {top1_per_layer.max():.3f}')
print(f'gold  range: {gold_per_layer.min():.3f} – {gold_per_layer.max():.3f}')
print(f'sum   range: {sum_per_layer.min():.3f} – {sum_per_layer.max():.3f}')
print(f'\nReady. {N} sentences across {len(EMOTIONS_23)} emotions.')

## Figura 20 — Logit lens vs probe

Tres paneles por grupo de cristalización (tempranas, medias, tardías).
Cada panel: probe F1 (rojo) y gold sigmoid del logit lens (verde) sobre
el mismo eje [0, 1]. La banda gris es información que el probe extrae
pero el classifier entrenado no usa.

In [ ]:
# Figura 20 — Logit lens vs probe (3 paneles por capa de cristalización)
layer_cols = ['Emb'] + [f'L{i}' for i in range(12)]
probe_df_indexed = (probe_df.set_index('emotion')[layer_cols]
                    if 'emotion' in probe_df.columns
                    else probe_df.set_index(probe_df.columns[0])[layer_cols])

emo_to_xlayer = dict(zip(crystal_df['emotion'],
                         crystal_df['crystallization_layer'].astype(int)))

def _bucket(layer):
    if layer <= 3: return 'early'
    if layer <= 7: return 'middle'
    return 'late'

emo_to_group = {e: _bucket(emo_to_xlayer.get(e, 11)) for e in EMOTIONS_23}

groups_data = {}
for g in ('early', 'middle', 'late'):
    emos = sorted([e for e, gg in emo_to_group.items() if gg == g])
    mask = np.isin(label_arr, emos)
    probe_subset = probe_df_indexed.loc[probe_df_indexed.index.isin(emos)]
    groups_data[g] = dict(
        emos=emos,
        n=int(mask.sum()),
        probe=probe_subset.mean(axis=0).values,
        gold=gold_sig[mask].mean(axis=0),
    )

print(f"early:  {len(groups_data['early']['emos'])} emos, {groups_data['early']['n']} frases")
print(f"middle: {len(groups_data['middle']['emos'])} emos, {groups_data['middle']['n']} frases")
print(f"late:   {len(groups_data['late']['emos'])} emos, {groups_data['late']['n']} frases")


def plot_lens_vs_probe(axes, groups=groups_data, lang='es'):
    titles_es = {
        'early':  'Tempranas (cristalizan en L0–L2)',
        'middle': 'Medias (cristalizan en L3–L6)',
        'late':   'Tardías (cristalizan en L7–L11)',
    }
    titles_en = {
        'early':  'Early (crystallize at L0–L2)',
        'middle': 'Middle (crystallize at L3–L6)',
        'late':   'Late (crystallize at L7–L11)',
    }
    titles = titles_es if lang == 'es' else titles_en
    n_lbl    = 'frases' if lang == 'es' else 'sentences'
    emo_lbl  = 'emociones' if lang == 'es' else 'emotions'
    probe_lbl = 'Probe F1'
    gold_lbl  = 'Gold sigmoid (logit lens)'
    gap_lbl   = 'Info no leída por el classifier' if lang == 'es' \
                else 'Info unread by the classifier'
    capa_lbl  = sty.L['layer']
    ylab      = 'F1 / sigmoid'

    layer_x = np.arange(13)

    for ax, g in zip(axes, ('early', 'middle', 'late')):
        d = groups[g]

        ax.axvspan(-0.5, 3.5, alpha=0.06, color=sty.SAND, zorder=0)
        ax.axvspan(3.5, 9.5,  alpha=0.04, color=sty.BLUE, zorder=0)
        ax.axvspan(9.5, 12.5, alpha=0.06, color=sty.TERRA, zorder=0)

        gap_top = np.maximum(d['probe'], d['gold'])
        gap_bot = np.minimum(d['probe'], d['gold'])
        ax.fill_between(layer_x, gap_bot, gap_top, color=sty.INK_3,
                        alpha=0.12, zorder=1,
                        label=gap_lbl if g == 'early' else None)

        ax.plot(layer_x, d['probe'], color=sty.TERRA, marker='s', markersize=5,
                lw=1.8, markeredgecolor='white', markeredgewidth=0.8,
                label=probe_lbl if g == 'early' else None, zorder=3)
        ax.plot(layer_x, d['gold'], color=sty.SAGE, marker='o', markersize=5,
                lw=1.8, markeredgecolor='white', markeredgewidth=0.8,
                label=gold_lbl if g == 'early' else None, zorder=3)

        title = f"{titles[g]}  ·  {len(d['emos'])} {emo_lbl} · {d['n']} {n_lbl}"
        ax.set_title(title, fontsize=sty.LEGEND_SIZE, color=sty.INK_2,
                     loc='left', pad=4)
        ax.set_ylim(-0.02, 1.02)
        ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
        ax.set_yticklabels(['0', '0.25', '0.5', '0.75', '1'])
        ax.grid(axis='y', alpha=0.3)
        ax.set_xticks(layer_x)
        ax.set_xticklabels([])
        ax.set_xlim(-0.5, 12.5)

    axes[0].legend(loc='upper left', fontsize=sty.LEGEND_SIZE,
                   ncol=3, framealpha=0.92)
    axes[1].set_ylabel(ylab, fontsize=sty.LABEL_SIZE, color=sty.INK_2)
    axes[-1].set_xticklabels(layer_cols, fontsize=sty.TICK_SIZE)
    axes[-1].set_xlabel(capa_lbl, fontsize=sty.LABEL_SIZE, color=sty.INK_2)


for lang in ['es', 'en']:
    sty.apply(lang=lang)
    fig, axes = plt.subplots(3, 1, figsize=(7.2, 7.0), sharex=False)
    plot_lens_vs_probe(axes, lang=lang)
    plt.tight_layout()
    sty.save(fig, 'lens_vs_probe', chapter=5, lang=lang)

## Figura 21 — Logit lens agregado · la curva en U

Aplicando el classifier real del modelo a cada capa (logit lens) sobre las
≈ 2.300 frases del test, las tres curvas trazan un patrón en U bien definido:

- **top-1 sigmoid** — probabilidad media de la emoción más probable.
- **gold sigmoid** — probabilidad media de la emoción correcta.
- **suma de 23 sigmoides** — masa total de probabilidad a través de las
  emociones (eje Y derecho).

La forma en U distingue tres regímenes geométricos sucesivos: saturación
(Emb-L3, donde el tanh del pooler satura ±1 y dispara muchas emociones a
la vez), valle de transición (L4-L9, donde el [CLS] no está alineado con
el espacio del classifier y todas las sigmoides colapsan), y cristalización
(L10-L11, donde la decisión emerge nítidamente).

In [ ]:
# Figura 21 — Logit lens U-curve agregado
def plot_iterative_u(lang='es'):
    sty.apply(lang=lang)
    fig, ax = plt.subplots(figsize=(7.2, 4.2))

    L = {
        'es': dict(top1='top-1 sigmoid', gold='gold sigmoid',
                   sum='suma de 23',
                   sat='saturación', valley='valle de transición',
                   crys='cristalización',
                   ylabel='sigmoid promedio',
                   ylabel2='suma sigmoid (de 23)',
                   xlabel=sty.L['layer'],
                   note=f'agregado sobre {N:,} frases del test set'.replace(',', '.')),
        'en': dict(top1='top-1 sigmoid', gold='gold sigmoid',
                   sum='sum of 23',
                   sat='saturation', valley='transition valley',
                   crys='crystallization',
                   ylabel='mean sigmoid',
                   ylabel2='sum of sigmoids (of 23)',
                   xlabel='Layer',
                   note=f'aggregated over {N:,} test sentences'),
    }[lang]

    layer_x = np.arange(13)
    layer_cols_local = ['Emb'] + [f'L{i}' for i in range(12)]

    # Phase shading
    ax.axvspan(-0.5, 3.5, alpha=0.10, color=sty.SAND, zorder=0)
    ax.axvspan(3.5, 9.5,  alpha=0.06, color=sty.BLUE, zorder=0)
    ax.axvspan(9.5, 12.5, alpha=0.10, color=sty.TERRA, zorder=0)

    # Phase labels
    ax.text(1.5, 0.80, L['sat'], ha='center', color=sty.SAND,
            fontsize=sty.ANNOTATION_SIZE, style='italic',
            fontweight='bold', zorder=4)
    ax.text(6.5, 0.80, L['valley'], ha='center', color=sty.BLUE,
            fontsize=sty.ANNOTATION_SIZE, style='italic',
            fontweight='bold', zorder=4)
    ax.text(11.0, 0.80, L['crys'], ha='center', color=sty.TERRA,
            fontsize=sty.ANNOTATION_SIZE, style='italic',
            fontweight='bold', zorder=4)

    # Left axis: top-1 + gold
    ax.plot(layer_x, top1_per_layer, color=sty.TERRA, marker='s',
            markersize=6, lw=2.2, markeredgecolor='white',
            markeredgewidth=0.9, label=L['top1'], zorder=5)
    ax.plot(layer_x, gold_per_layer, color=sty.SAGE, marker='o',
            markersize=6, lw=2.0, markeredgecolor='white',
            markeredgewidth=0.9, label=L['gold'], zorder=5)

    ax.set_xlim(-0.5, 12.5)
    ax.set_ylim(0, 0.92)
    ax.set_xticks(layer_x)
    ax.set_xticklabels(layer_cols_local)
    ax.set_xlabel(L['xlabel'])
    ax.set_ylabel(L['ylabel'])
    ax.grid(axis='y', alpha=0.3)

    # Right axis: sum
    ax2 = ax.twinx()
    ax2.plot(layer_x, sum_per_layer, color=sty.INK_3, lw=1.2, ls=':',
             marker='^', markersize=4.5, markeredgecolor='white',
             markeredgewidth=0.6, label=L['sum'], zorder=4)
    ax2.set_ylim(0, 7.5)
    ax2.set_ylabel(L['ylabel2'], color=sty.INK_3, fontsize=sty.LABEL_SIZE)
    ax2.tick_params(axis='y', colors=sty.INK_3)
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(True)
    ax2.spines['right'].set_color(sty.SPINE)
    ax2.grid(False)

    # Combined legend
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, loc='upper left',
              fontsize=sty.LEGEND_SIZE, framealpha=0.92, ncol=1)

    # Sample size note
    ax.text(0.99, 0.02, L['note'], transform=ax.transAxes, ha='right',
            va='bottom', fontsize=sty.SMALL_SIZE, color=sty.INK_3,
            style='italic')

    plt.tight_layout()
    sty.save(fig, 'iterative_u', chapter=5, lang=lang)


for lang in ['es', 'en']:
    plot_iterative_u(lang)

## Resumen

Figuras generadas en `figures/cap5_*_{es,en}.png`. Los títulos se delegan al
`\caption{}` de LaTeX en el TFG.